# LeetCode 2176 — Count Equal and Divisible Pairs in an Array bằng Python

**Link đề:** https://leetcode.com/problems/count-equal-and-divisible-pairs-in-an-array/
**Độ khó:** Easy · **Chủ đề:** Array

```python
class Solution:
    def countPairs(self, nums: list[int], k: int) -> int:
        ...
```

## Mục tiêu học
1. Duyệt **mọi cặp chỉ số** `i < j` bằng hai vòng lồng nhau — kỹ thuật nền tảng.
2. Kiểm tra hai điều kiện cùng lúc: **giá trị bằng nhau** và **tích chỉ số chia hết cho `k`** (`(i * j) % k == 0`).
3. Tối ưu thực tế: **gom chỉ số theo giá trị** bằng `dict` / `defaultdict(list)` để chỉ xét cặp trong cùng nhóm.
4. Ý tưởng nâng cao: dùng **ƯCLN (gcd)** — `k | i·j` ⟺ `k | gcd(i,k)·gcd(j,k)` — để đếm theo nhóm ước.
5. Kiểm thử đối chiếu các cách và tránh lỗi đếm trùng `(i, j)` / `(j, i)`.

## 1. Đề bài

Cho mảng `nums` (chỉ số từ 0) độ dài `n` và số nguyên `k`. Đếm số cặp `(i, j)` với `0 <= i < j < n` sao cho:
- `nums[i] == nums[j]`, **và**
- `i * j` chia hết cho `k`.

| `nums` | `k` | Kết quả | Các cặp |
|---|:-:|:-:|---|
| `[3,1,2,2,2,1,3]` | 2 | `4` | (0,6): 0·6=0; (2,3): 6; (2,4): 8; (3,4): 12 |
| `[1,2,3,4]` | 1 | `0` | không có giá trị lặp |

**Constraints:** `1 <= len(nums) <= 100`, `1 <= nums[i], k <= 100`.

Chú ý: điều kiện chia hết áp dụng cho **chỉ số** `i, j`, không phải giá trị. Và `0` chia hết cho mọi `k` → mọi cặp có `i = 0` thỏa điều kiện thứ hai.

In [1]:
nums, k = [3, 1, 2, 2, 2, 1, 3], 2
for i, x in enumerate(nums):
    print(f"chỉ số {i}: giá trị {x}")

chỉ số 0: giá trị 3
chỉ số 1: giá trị 1
chỉ số 2: giá trị 2
chỉ số 3: giá trị 2
chỉ số 4: giá trị 2
chỉ số 5: giá trị 1
chỉ số 6: giá trị 3


## 2. Nền tảng Python cần biết

### 2.1. Hai vòng lồng nhau duyệt mọi cặp `i < j`

`for i in range(n)` ngoài và `for j in range(i + 1, n)` trong → mỗi cặp không thứ tự xuất hiện **đúng một lần**, và không có `i == j`. Số cặp là `n·(n-1)/2`.

In [2]:
n = 4
pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
print(pairs)
print("số cặp:", len(pairs), "= n(n-1)/2 =", n * (n - 1) // 2)

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
số cặp: 6 = n(n-1)/2 = 6


### 2.2. Chia hết: `a % k == 0`

`a % k` là số dư khi chia `a` cho `k`; dư `0` ⟺ `a` chia hết cho `k`. Đặc biệt `0 % k == 0`.

In [3]:
k = 2
for a in (0, 6, 8, 12, 5):
    print(f"{a} % {k} = {a % k} -> {'chia hết' if a % k == 0 else 'không'}")

0 % 2 = 0 -> chia hết
6 % 2 = 0 -> chia hết
8 % 2 = 0 -> chia hết
12 % 2 = 0 -> chia hết
5 % 2 = 1 -> không


### 2.3. `dict` gom nhóm và `collections.defaultdict(list)`

- `groups = {}` rồi `groups.setdefault(x, []).append(i)` — tạo list rỗng nếu chưa có key `x`.
- `defaultdict(list)` làm việc đó tự động: truy cập key chưa có sẽ tạo `[]`.

In [4]:
from collections import defaultdict
nums = [3, 1, 2, 2, 2, 1, 3]
groups = defaultdict(list)
for i, x in enumerate(nums):
    groups[x].append(i)
print(dict(groups))

{3: [0, 6], 1: [1, 5], 2: [2, 3, 4]}


### 2.4. `math.gcd` — ước chung lớn nhất

`gcd(a, b)` là số lớn nhất chia hết cả `a` và `b`. `gcd(0, k) = k`. Ta sẽ dùng ở Cách 4.

In [5]:
from math import gcd
print(gcd(6, 4), gcd(0, 4), gcd(3, 4), gcd(8, 12))

2 4 1 4


### 2.5. `itertools.combinations` — sinh cặp gọn

`combinations(seq, 2)` sinh mọi cặp `(seq[a], seq[b])` với `a < b` — tương đương hai vòng lồng nhau.

In [6]:
from itertools import combinations
print(list(combinations([2, 3, 4], 2)))

[(2, 3), (2, 4), (3, 4)]


## 3. Cách 1 — Brute force: thử mọi cặp (lời giải chuẩn cho constraints)

Với `n <= 100`, chỉ có tối đa `4950` cặp → brute force là lời giải **hoàn toàn hợp lý** (hint của đề cũng gợi ý vậy).

Từng đoạn:

**Đoạn 1:** `for i in range(n)`, `for j in range(i + 1, n)`.

**Đoạn 2:** kiểm tra `nums[i] == nums[j]` (rẻ, lọc trước).

**Đoạn 3:** kiểm tra `(i * j) % k == 0`.

**Đoạn 4:** thỏa cả hai → `count += 1`.

In [7]:
def count_pairs_bruteforce(nums: list[int], k: int) -> int:
    n = len(nums)
    count = 0
    for i in range(n):                                   # Đoạn 1
        for j in range(i + 1, n):
            if nums[i] == nums[j] and (i * j) % k == 0:  # Đoạn 2 + 3
                count += 1                               # Đoạn 4
    return count

print(count_pairs_bruteforce([3, 1, 2, 2, 2, 1, 3], 2))   # 4
print(count_pairs_bruteforce([1, 2, 3, 4], 1))            # 0

4
0


`and` trong Python **dừng sớm** (short-circuit): nếu `nums[i] != nums[j]` thì phép `%` không được tính. Đặt điều kiện rẻ/hay sai trước là thói quen tốt.

## 4. TRACE từng bước

`nums = [3,1,2,2,2,1,3]`, `k = 2`. Chỉ in các cặp **giá trị bằng nhau** (các cặp khác bị loại ngay ở điều kiện đầu):

| `(i, j)` | giá trị | `i * j` | `% 2` | Đếm? | `count` |
|---|:-:|:-:|:-:|:-:|:-:|
| (0, 6) | 3 | 0 | 0 | ✓ | 1 |
| (1, 5) | 1 | 5 | 1 | ✗ | 1 |
| (2, 3) | 2 | 6 | 0 | ✓ | 2 |
| (2, 4) | 2 | 8 | 0 | ✓ | 3 |
| (3, 4) | 2 | 12 | 0 | ✓ | 4 |

In [8]:
def trace_count_pairs(nums, k):
    n = len(nums)
    count = 0
    checked = 0
    for i in range(n):
        for j in range(i + 1, n):
            checked += 1
            if nums[i] != nums[j]:
                continue
            ok = (i * j) % k == 0
            if ok:
                count += 1
            print(f"(i={i}, j={j}) giá trị {nums[i]} | i*j={i * j:>2} | %{k}={(i * j) % k} -> {'đếm' if ok else 'bỏ'} | count={count}")
    print(f"Đã xét {checked} cặp, kết quả = {count}")
    return count

trace_count_pairs([3, 1, 2, 2, 2, 1, 3], 2)

(i=0, j=6) giá trị 3 | i*j= 0 | %2=0 -> đếm | count=1
(i=1, j=5) giá trị 1 | i*j= 5 | %2=1 -> bỏ | count=1
(i=2, j=3) giá trị 2 | i*j= 6 | %2=0 -> đếm | count=2
(i=2, j=4) giá trị 2 | i*j= 8 | %2=0 -> đếm | count=3
(i=3, j=4) giá trị 2 | i*j=12 | %2=0 -> đếm | count=4
Đã xét 21 cặp, kết quả = 4


4

## 5. Code nộp LeetCode — Cách 1

In [9]:
class Solution:
    def countPairs(self, nums: list[int], k: int) -> int:
        n = len(nums)
        count = 0
        for i in range(n):
            for j in range(i + 1, n):
                if nums[i] == nums[j] and i * j % k == 0:
                    count += 1
        return count

sol = Solution()
print(sol.countPairs([3, 1, 2, 2, 2, 1, 3], 2), sol.countPairs([1, 2, 3, 4], 1))

4 0


`i * j % k` được hiểu là `(i * j) % k` vì `*` và `%` **cùng độ ưu tiên**, tính từ trái sang phải. Viết thêm ngoặc cho dễ đọc vẫn tốt hơn.

In [10]:
i, j, k = 3, 4, 5
print(i * j % k, (i * j) % k, i * (j % k))

2 2 12


## 6. Nút thắt và insight

Brute force xét **mọi** cặp, kể cả cặp giá trị khác nhau — chúng chắc chắn bị loại. Với mảng nhiều giá trị khác nhau, phần lớn công sức bị lãng phí.

**Insight:** chỉ cặp **cùng giá trị** mới có cơ hội. Gom chỉ số theo giá trị trước (`dict`), rồi chỉ xét cặp **trong từng nhóm**.

- Nếu mọi giá trị khác nhau: mỗi nhóm có 1 chỉ số → **0** cặp cần xét.
- Nếu mọi giá trị bằng nhau: một nhóm `n` chỉ số → vẫn `n(n-1)/2` cặp (worst case không đổi).

Đếm thử số cặp phải xét:

In [11]:
def pairs_to_check_grouped(nums):
    groups = defaultdict(list)
    for i, x in enumerate(nums):
        groups[x].append(i)
    return sum(len(g) * (len(g) - 1) // 2 for g in groups.values())

for nums in ([3, 1, 2, 2, 2, 1, 3], list(range(1, 101)), [7] * 100):
    n = len(nums)
    print(f"n={n:>3}: brute force {n * (n - 1) // 2:>4} cặp | gom nhóm {pairs_to_check_grouped(nums):>4} cặp")

n=  7: brute force   21 cặp | gom nhóm    5 cặp
n=100: brute force 4950 cặp | gom nhóm    0 cặp
n=100: brute force 4950 cặp | gom nhóm 4950 cặp


## 7. Cách 2 — Gom chỉ số theo giá trị (dict of lists)

Từng đoạn:

**Đoạn 1:** `groups[x].append(i)` cho mỗi `i, x`.

**Đoạn 2:** với mỗi nhóm `idx` (list chỉ số tăng dần), duyệt cặp `a < b` trong nhóm.

**Đoạn 3:** đếm nếu `idx[a] * idx[b] % k == 0`.

In [12]:
def count_pairs_grouped(nums: list[int], k: int) -> int:
    groups = defaultdict(list)
    for i, x in enumerate(nums):                  # Đoạn 1
        groups[x].append(i)
    count = 0
    for idx in groups.values():                   # Đoạn 2
        for a in range(len(idx)):
            for b2 in range(a + 1, len(idx)):
                if idx[a] * idx[b2] % k == 0:     # Đoạn 3
                    count += 1
    return count

print(count_pairs_grouped([3, 1, 2, 2, 2, 1, 3], 2), count_pairs_grouped([1, 2, 3, 4], 1))

4 0


In [13]:
def trace_grouped(nums, k):
    groups = defaultdict(list)
    for i, x in enumerate(nums):
        groups[x].append(i)
    total = 0
    for value, idx in groups.items():
        good = [(i, j) for i, j in combinations(idx, 2) if i * j % k == 0]
        total += len(good)
        print(f"giá trị {value}: chỉ số {idx} -> cặp thỏa {good} (+{len(good)})")
    print("tổng =", total)
    return total

trace_grouped([3, 1, 2, 2, 2, 1, 3], 2)

giá trị 3: chỉ số [0, 6] -> cặp thỏa [(0, 6)] (+1)
giá trị 1: chỉ số [1, 5] -> cặp thỏa [] (+0)
giá trị 2: chỉ số [2, 3, 4] -> cặp thỏa [(2, 3), (2, 4), (3, 4)] (+3)
tổng = 4


4

In [14]:
class Solution:
    def countPairs(self, nums: list[int], k: int) -> int:
        groups = defaultdict(list)
        for i, x in enumerate(nums):
            groups[x].append(i)
        return sum(1 for idx in groups.values()
                     for i, j in combinations(idx, 2)
                     if i * j % k == 0)

print(Solution().countPairs([3, 1, 2, 2, 2, 1, 3], 2))

4


(Trên LeetCode, `defaultdict` và `combinations` đã có sẵn trong môi trường Python3; khi tự chạy ngoài, nhớ `from collections import defaultdict` và `from itertools import combinations`.)

## 8. Cách 3 — Một lượt duyệt, so với các chỉ số **trước đó** cùng giá trị

Thay vì gom xong rồi mới ghép cặp, ta duyệt `j` từ trái sang phải; với mỗi `j`, so với mọi `i` đã thấy trước đó có cùng giá trị (đã lưu trong `seen[x]`), rồi thêm `j` vào. Đây là khuôn mẫu "**đếm cặp khi duyệt: nhìn lại quá khứ**" rất phổ biến (giống Two Sum).

In [15]:
def count_pairs_online(nums: list[int], k: int) -> int:
    seen = defaultdict(list)
    count = 0
    for j, x in enumerate(nums):
        for i in seen[x]:                 # các i < j cùng giá trị
            if i * j % k == 0:
                count += 1
        seen[x].append(j)                 # kiểm tra trước, lưu sau
    return count

print(count_pairs_online([3, 1, 2, 2, 2, 1, 3], 2))

4


## 9. Cách 4 — Nâng cao: đếm theo ƯCLN với `k`

**Bổ đề:** `k | i·j` ⟺ `k | gcd(i, k) · gcd(j, k)`.

Lý do trực giác: phần của `i` "có ích" cho việc chia hết `k` chỉ là `gcd(i, k)` — các thừa số nguyên tố của `i` không nằm trong `k` không giúp gì. (Chiều ⇐: `gcd(i,k)·gcd(j,k)` là ước của `i·j`. Chiều ⇒: xét từng thừa số nguyên tố `p` của `k` với số mũ `e`: `v_p(i) + v_p(j) >= e` ⇒ `min(v_p(i), e) + min(v_p(j), e) >= e`.)

Vậy với mỗi giá trị, thay vì nhớ từng chỉ số, chỉ cần nhớ **đếm theo `g = gcd(i, k)`**. Số giá trị `g` khác nhau ≤ số ước của `k` (với `k ≤ 100` là tối đa 12).

In [16]:
k = 12
for i, j in [(3, 4), (2, 6), (6, 8), (5, 12), (4, 9)]:
    lhs = (i * j) % k == 0
    rhs = (gcd(i, k) * gcd(j, k)) % k == 0
    print(f"i={i}, j={j}: k|i*j={lhs}, k|gcd(i,k)*gcd(j,k)={rhs} (gcd: {gcd(i, k)}, {gcd(j, k)})")

i=3, j=4: k|i*j=True, k|gcd(i,k)*gcd(j,k)=True (gcd: 3, 4)
i=2, j=6: k|i*j=True, k|gcd(i,k)*gcd(j,k)=True (gcd: 2, 6)
i=6, j=8: k|i*j=True, k|gcd(i,k)*gcd(j,k)=True (gcd: 6, 4)
i=5, j=12: k|i*j=True, k|gcd(i,k)*gcd(j,k)=True (gcd: 1, 12)
i=4, j=9: k|i*j=True, k|gcd(i,k)*gcd(j,k)=True (gcd: 4, 3)


In [17]:
from collections import Counter

def count_pairs_gcd(nums: list[int], k: int) -> int:
    seen = defaultdict(Counter)          # value -> Counter{g: số chỉ số i có gcd(i,k)=g}
    count = 0
    for j, x in enumerate(nums):
        gj = gcd(j, k)
        for g, c in seen[x].items():
            if g * gj % k == 0:
                count += c
        seen[x][gj] += 1
    return count

print(count_pairs_gcd([3, 1, 2, 2, 2, 1, 3], 2), count_pairs_gcd([1, 2, 3, 4], 1))

4 0


`defaultdict(Counter)`: mỗi giá trị `x` có một `Counter` riêng (một dict đếm, key chưa có thì mặc định 0). Độ phức tạp `O(n · d(k))` với `d(k)` là số ước của `k` — đây chính là ý tưởng của bài khó hơn **#2183 Count Array Pairs Divisible by K**.

## 10. Vì sao đúng? (invariant)

**Cách 1:** hai vòng sinh mỗi cặp `i < j` đúng một lần; điều kiện `if` chính là định nghĩa của đề ⇒ đếm đúng.

**Cách 2:** cặp hợp lệ phải có `nums[i] == nums[j]` ⇒ `i`, `j` thuộc cùng nhóm. Trong nhóm, chỉ số được thêm theo thứ tự tăng, nên `a < b` ⟺ `idx[a] < idx[b]`. Mỗi cặp hợp lệ được đếm đúng một lần ở nhóm của nó; cặp khác nhóm bị loại đúng.

**Cách 3 — invariant:** trước khi xử lý `j`, `seen[x]` là danh sách mọi `i < j` có `nums[i] == x`. Vì "kiểm tra trước, lưu sau", `j` không bao giờ ghép với chính nó.

**Cách 4:** như Cách 3, nhưng nhờ bổ đề, điều kiện chỉ phụ thuộc `gcd(i, k)` nên ta gộp các `i` cùng `gcd` lại bằng cách đếm.

## 11. Độ phức tạp

Gọi `n = len(nums)`, `g_v` là số chỉ số có giá trị `v`, `d(k)` là số ước của `k`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n²)` | `O(1)` | lời giải chuẩn với `n ≤ 100` |
| 2. Gom nhóm | `O(n + Σ g_v²)`, worst `O(n²)` | `O(n)` | nhanh khi ít giá trị trùng |
| 3. Một lượt + nhìn lại | như cách 2 | `O(n)` | khuôn mẫu đếm cặp online |
| 4. Đếm theo gcd | `O(n · (d(k) + log k))` | `O(n)` | nâng cao, mở đường cho #2183 |

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

1. Hai ví dụ của đề.
2. `n = 1` (không có cặp).
3. `k = 1` (mọi tích chia hết) → chỉ còn đếm cặp cùng giá trị.
4. Mọi phần tử bằng nhau.
5. Chỉ cặp có `i = 0` thỏa (ví dụ `k` lớn).

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS ({actual})")

impls = [count_pairs_bruteforce, count_pairs_grouped, count_pairs_online, count_pairs_gcd, Solution().countPairs]
cases = [
    ("đề ví dụ 1", [3, 1, 2, 2, 2, 1, 3], 2, 4),
    ("đề ví dụ 2", [1, 2, 3, 4], 1, 0),
    ("n = 1", [5], 3, 0),
    ("k = 1, toàn bằng nhau", [7, 7, 7, 7], 1, 6),
    ("toàn bằng nhau, k = 2", [7, 7, 7, 7], 2, 5),     # (0,1),(0,2),(0,3),(1,2),(2,3)
    ("chỉ i = 0 thỏa", [9, 9, 9], 100, 2),
    ("k lớn hơn mọi tích", [1, 1], 100, 1),            # (0,1): 0 chia hết
    ("cặp không chứa 0", [5, 1, 1, 1], 6, 1),          # (1,2)=2,(1,3)=3,(2,3)=6
]
for name, nums, k, exp in cases:
    for f in impls:
        assert f(list(nums), k) == exp, (name, f)
    check(name, count_pairs_bruteforce(nums, k), exp)

đề ví dụ 1                 PASS (4)
đề ví dụ 2                 PASS (0)
n = 1                      PASS (0)
k = 1, toàn bằng nhau      PASS (6)
toàn bằng nhau, k = 2      PASS (5)
chỉ i = 0 thỏa             PASS (2)
k lớn hơn mọi tích         PASS (1)
cặp không chứa 0           PASS (1)


In [19]:
import random
rng = random.Random(2176)
for _ in range(500):
    n = rng.randint(1, 40)
    nums = [rng.randint(1, 4) for _ in range(n)]
    k = rng.randint(1, 30)
    expected = count_pairs_bruteforce(nums, k)
    for f in impls:
        assert f(list(nums), k) == expected, (nums, k, f)
print("500 test ngẫu nhiên: mọi cách khớp brute force")

500 test ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1: Kiểm tra chia hết trên **giá trị** thay vì **chỉ số**

In [20]:
def wrong_values(nums, k):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + 1, n)
               if nums[i] == nums[j] and nums[i] * nums[j] % k == 0)

print("sai :", wrong_values([3, 1, 2, 2, 2, 1, 3], 2), "| đúng:", count_pairs_bruteforce([3, 1, 2, 2, 2, 1, 3], 2))

sai : 3 | đúng: 4


**Vì sao sai:** đề yêu cầu `(i * j)` chia hết cho `k` — `i`, `j` là **chỉ số**. **Sửa:** dùng `i * j`.

### Lỗi 2: Vòng trong bắt đầu từ `0` → đếm trùng và đếm cả `i == j`

In [21]:
def wrong_double_count(nums, k):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(n)
               if nums[i] == nums[j] and i * j % k == 0)

print("sai :", wrong_double_count([3, 1, 2, 2, 2, 1, 3], 2), "| đúng:", 4)

sai : 12 | đúng: 4


**Vì sao sai:** đếm cả `(j, i)` lẫn `(i, j)`, và cặp `(i, i)` (ví dụ `(0, 0)`, `(2, 2)`...). **Sửa:** `for j in range(i + 1, n)`.

### Lỗi 3: Quên rằng `i = 0` luôn thỏa điều kiện chia hết
Một số bạn "tối ưu" bằng cách bắt đầu `i` từ 1 vì nghĩ chỉ số 0 "đặc biệt":

In [22]:
def wrong_skip_zero(nums, k):
    n = len(nums)
    return sum(1 for i in range(1, n) for j in range(i + 1, n)
               if nums[i] == nums[j] and i * j % k == 0)

print("sai :", wrong_skip_zero([3, 1, 2, 2, 2, 1, 3], 2), "| đúng:", 4)

sai : 3 | đúng: 4


**Vì sao sai:** `0 * j = 0` chia hết mọi `k`, nên cặp `(0, j)` cùng giá trị luôn được đếm. **Sửa:** `i` bắt đầu từ `0`.

### Lỗi 4: Ở Cách 3, lưu `j` **trước** khi kiểm tra

In [23]:
def wrong_store_first(nums, k):
    seen = defaultdict(list)
    count = 0
    for j, x in enumerate(nums):
        seen[x].append(j)                   # lưu trước
        for i in seen[x]:
            if i * j % k == 0:
                count += 1
    return count

print("sai :", wrong_store_first([3, 1, 2, 2, 2, 1, 3], 2), "| đúng:", 4)

sai : 8 | đúng: 4


**Vì sao sai:** `j` ghép với chính nó (`j * j`), ví dụ `(0, 0)`, `(2, 2)`, `(4, 4)`, `(6, 6)` bị đếm thêm. **Sửa:** kiểm tra trước, lưu sau.

## 14. Mẹo nhận dạng pattern

- "Đếm cặp `i < j` thỏa điều kiện", `n` nhỏ (≤ 100, ≤ 1000) → brute force hai vòng là đủ.
- Điều kiện có **`nums[i] == nums[j]`** → gom chỉ số theo giá trị bằng `dict` để bỏ cặp vô ích.
- "Đếm khi duyệt": với mỗi `j`, nhìn lại những gì đã lưu cho `i < j`; **kiểm tra trước, lưu sau**.
- Điều kiện chia hết tích `k | a·b` → nghĩ tới `gcd(a, k)` để nén trạng thái.

Bài liên quan: #1512 Number of Good Pairs, #2006 Count Number of Pairs With Absolute Difference K, #2183 Count Array Pairs Divisible by K (Hard), #1010 Pairs of Songs With Total Durations Divisible by 60.

## 15. Biến thể

### 15.1. LeetCode 1512 — Number of Good Pairs (bỏ điều kiện chia hết)
Đếm cặp `i < j` có `nums[i] == nums[j]`: mỗi nhóm `c` phần tử đóng góp `c·(c-1)/2` — không cần duyệt cặp.

In [24]:
def num_identical_pairs(nums):
    return sum(c * (c - 1) // 2 for c in Counter(nums).values())

print(num_identical_pairs([1, 2, 3, 1, 1, 3]))   # 4
print(count_pairs_bruteforce([1, 2, 3, 1, 1, 3], 1), "<- k = 1 cho cùng kết quả")

4
4 <- k = 1 cho cùng kết quả


### 15.2. Điều kiện **tổng** chỉ số chia hết cho `k`
`(i + j) % k == 0` → nhóm theo `i % k`: cần `i % k + j % k ≡ 0 (mod k)`.

In [25]:
def count_pairs_sum_divisible(nums, k):
    seen = defaultdict(Counter)       # value -> Counter{i % k: số lượng}
    count = 0
    for j, x in enumerate(nums):
        need = (-j) % k
        count += seen[x][need]
        seen[x][j % k] += 1
    return count

def brute_sum(nums, k):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + 1, n) if nums[i] == nums[j] and (i + j) % k == 0)

print(count_pairs_sum_divisible([3, 1, 2, 2, 2, 1, 3], 2), brute_sum([3, 1, 2, 2, 2, 1, 3], 2))

3 3


## 16. Bài tập (TODO — tự làm, đáp án ở cuối)

### Bài 1 — Tính tay
`nums = [2, 2, 2, 2, 2]`, `k = 3`. Liệt kê các cặp `(i, j)` thỏa đề.

### Bài 2 — Đoán output
1. `len(list(combinations(range(6), 2)))` = ?
2. `gcd(0, 7)` = ?
3. `countPairs([1, 1], 5)` = ?

### Bài 3 — Điền chỗ trống (Cách 1)

In [26]:
def exercise_count_pairs(nums: list[int], k: int) -> int:
    count = 0
    # TODO 1: for i in range(n)
    # TODO 2: for j in range(i + 1, n)
    # TODO 3: nếu nums[i] == nums[j] và i * j % k == 0 thì count += 1
    return count

# assert exercise_count_pairs([3, 1, 2, 2, 2, 1, 3], 2) == 4

### Bài 4 — Trả về **danh sách cặp** thay vì số lượng

In [27]:
def exercise_list_pairs(nums: list[int], k: int) -> list[tuple[int, int]]:
    # TODO: trả về các (i, j) theo thứ tự i tăng, rồi j tăng
    pass

# assert exercise_list_pairs([3, 1, 2, 2, 2, 1, 3], 2) == [(0, 6), (2, 3), (2, 4), (3, 4)]

### Bài 5 — Biến thể: Number of Good Pairs bằng một lượt duyệt (không dùng công thức)
Duyệt `j`, cộng số lần `nums[j]` đã xuất hiện trước đó.

In [28]:
def exercise_good_pairs_online(nums: list[int]) -> int:
    # TODO: dict đếm; kiểm tra trước, lưu sau
    pass

# assert exercise_good_pairs_online([1, 2, 3, 1, 1, 3]) == 4
# assert exercise_good_pairs_online([1, 1, 1, 1]) == 6

### Bài 6 — Mini challenge: đếm cặp `i < j` (không cần giá trị bằng nhau) có `nums[i] * nums[j]` chia hết cho `k`
Đây là phiên bản rút gọn của #2183. Yêu cầu dùng ý tưởng gcd, `O(n · d(k))`.

In [29]:
def exercise_product_divisible(nums: list[int], k: int) -> int:
    # TODO: Counter theo gcd(x, k)
    pass

# assert exercise_product_divisible([1, 2, 3, 4, 5], 2) == 7
# assert exercise_product_divisible([1, 2, 3, 4], 5) == 0

### Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Mọi cặp cùng giá trị | cần `i·j` chia hết 3 ⇒ `i` hoặc `j` chia hết 3 | chỉ số chia hết 3: 0 và 3 |
| 2 | `C(6,2)` | `gcd(0, n) = n` | cặp `(0, 1)`: `0` chia hết `5` |
| 3 | Hai vòng | `range(i + 1, n)` | `and` hai điều kiện |
| 4 | Như bài 3 | thay `count += 1` bằng `append((i, j))` | thứ tự tự nhiên của hai vòng |
| 5 | `freq = {}` | `count += freq.get(x, 0)` | `freq[x] = freq.get(x, 0) + 1` sau đó |
| 6 | Bổ đề gcd | `Counter` của `gcd(x, k)` cho phần đã thấy | với `x` mới: cộng `cnt[g]` nếu `g * gcd(x,k) % k == 0` |

## 17. Đáp án tham khảo (tự làm trước rồi hãy xem)

**Bài 1:** `k = 3` là số nguyên tố nên `3 | i·j` ⟺ `3 | i` hoặc `3 | j`. Chỉ số chia hết cho 3 trong `0..4`: `{0, 3}`. Các cặp: `(0,1),(0,2),(0,3),(0,4),(1,3),(2,3),(3,4)` → **7** cặp.

**Bài 2:** `15`; `7`; `1`.

In [30]:
print(count_pairs_bruteforce([2, 2, 2, 2, 2], 3), [(i, j) for i, j in combinations(range(5), 2) if i * j % 3 == 0])
print(len(list(combinations(range(6), 2))), gcd(0, 7), Solution().countPairs([1, 1], 5))

7 [(0, 1), (0, 2), (0, 3), (0, 4), (1, 3), (2, 3), (3, 4)]
15 7 1


In [31]:
def answer_count_pairs(nums, k):
    n = len(nums)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] == nums[j] and i * j % k == 0:
                count += 1
    return count


def answer_list_pairs(nums, k):
    n = len(nums)
    return [(i, j) for i in range(n) for j in range(i + 1, n)
            if nums[i] == nums[j] and i * j % k == 0]


def answer_good_pairs_online(nums):
    freq = {}
    count = 0
    for x in nums:
        count += freq.get(x, 0)
        freq[x] = freq.get(x, 0) + 1
    return count


def answer_product_divisible(nums, k):
    cnt = Counter()
    count = 0
    for x in nums:
        gx = gcd(x, k)
        for g, c in cnt.items():
            if g * gx % k == 0:
                count += c
        cnt[gx] += 1
    return count


def brute_product(nums, k):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + 1, n) if nums[i] * nums[j] % k == 0)


assert answer_count_pairs([3, 1, 2, 2, 2, 1, 3], 2) == 4
assert answer_list_pairs([3, 1, 2, 2, 2, 1, 3], 2) == [(0, 6), (2, 3), (2, 4), (3, 4)]
assert answer_good_pairs_online([1, 2, 3, 1, 1, 3]) == 4
assert answer_good_pairs_online([1, 1, 1, 1]) == 6
assert answer_product_divisible([1, 2, 3, 4, 5], 2) == 7
assert answer_product_divisible([1, 2, 3, 4], 5) == 0
for _ in range(300):
    nums = [rng.randint(1, 50) for _ in range(rng.randint(1, 30))]
    k = rng.randint(1, 60)
    assert answer_product_divisible(nums, k) == brute_product(nums, k)
print("Đáp án tham khảo: PASS toàn bộ test!")

Đáp án tham khảo: PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countPairs(self, nums: list[int], k: int) -> int:
        n = len(nums)
        return sum(1 for i in range(n) for j in range(i + 1, n)
                   if nums[i] == nums[j] and i * j % k == 0)
```

**Khung tư duy đếm cặp:** brute force `i < j` khi `n` nhỏ → lọc bằng nhóm giá trị (`dict`) → đếm online "kiểm tra trước, lưu sau" → nén trạng thái (gcd, phần dư) khi điều kiện là chia hết.

**Checklist trước khi nộp:**
- [ ] Vòng trong bắt đầu từ `i + 1` (không trùng, không `i == j`)?
- [ ] Chia hết áp dụng cho **chỉ số** `i * j`, không phải giá trị?
- [ ] Không bỏ chỉ số `0`?
- [ ] Test `k = 1`, `n = 1`, mọi phần tử bằng nhau.